# Building the Backtester 

In [1]:
import pandas as pd
import numpy as np

#### Backtester Function

In [2]:
price = [10, 15, 12, 13, 20, 21, 18, 19, 25, 22]
signal = [0, 1, 1, 0, 1, 0, 0, 1, 1, 1]
transaction_cost = 0.001

In [3]:
def calculate_return(price):

    returns = [None]

    for i in range(1,len(price)):
        daily_return = (price[i] - price[i-1])/price[i-1]
        returns.append(daily_return)

    return returns

In [4]:
returns = calculate_return(price)

In [5]:
def backtest(price, signal, transaction_cost):

    returns = [None]

    for i in range(1,len(price)):
        daily_return = (price[i] - price[i-1])/price[i-1]
        returns.append(daily_return)

    df = pd.DataFrame({
        "return": returns,
        "signal": signal
    })

    df["position"] = df["signal"].shift(1)
    df["strategy_return"] = df["return"] * df["position"]


    # Time Invested

    days_invested = (df["position"] == 1).sum()
    valid_days = df["position"].count()
    time_invested = days_invested / valid_days


    # Trade Counts

    position = df["position"].fillna(0).tolist()
    trade_indicator = [0]
    
    for i in range(1, len(position)):
        if position[i] != position[i-1]:
            trade_indicator.append(1)
        else:
            trade_indicator.append(0)
    
    df["trade"] = trade_indicator
    trade_count = df["trade"].sum()


    # Transaction Costs

    df["transaction_cost"] = df["trade"] * transaction_cost
    df["net_strategy_return"] = df["strategy_return"] - df["transaction_cost"]


    # Equity

    performance_return = df["net_strategy_return"]

    initial_equity = 1.0
    df.loc[0, "equity"] = initial_equity
    
    for i in range(1,len(df["equity"])):
        df.loc[i, "equity"] = df.loc[i-1, "equity"] * (1 + performance_return[i])

    final_equity = float(df.loc[len(df)-1, "equity"])
    total_return = final_equity/initial_equity - 1


    # Volatility + Sharpe Ratio

    risk_free_rate = 0
    
    daily_volatility = performance_return.std()
    mean_daily_return = performance_return.mean()
    sharpe_daily = (mean_daily_return - risk_free_rate)/daily_volatility
    
    annual_volatility = np.sqrt(252) * daily_volatility
    sharpe_ratio = np.sqrt(252) * sharpe_daily


    # Drawdown

    df.loc[0, "running_peak"] = df.loc[0, "equity"]
    
    for i in range(1, len(df["equity"])):
        df.loc[i, "running_peak"] = max(df.loc[i, "equity"], df.loc[i-1, "running_peak"])
    
    df["drawdown"] = df["equity"] / df["running_peak"] - 1
    max_drawdown = df["drawdown"].min()


    # CAGR + Calmar Ratio

    growth_multiple = final_equity / initial_equity
    years = performance_return.count() / 252
    cagr = growth_multiple ** (1 / years) - 1
    calmar_ratio = cagr / abs(max_drawdown)


    results = {
        "total_return": total_return,
        "cagr": cagr,
        "annual_volatility": annual_volatility,
        "annual_sharpe": sharpe_ratio,
        "max_drawdown": max_drawdown,
        "calmar": calmar_ratio,
        "time_invested": time_invested,
        "trades": trade_count
    }

    return df, results
    

In [6]:
%%timeit 

returns = [None]

for i in range(1,len(price)):
    daily_return = (price[i] - price[i-1])/price[i-1]
    returns.append(daily_return)

df = pd.DataFrame({
    "return": returns,
    "signal": signal
})

df["position"] = df["signal"].shift(1)
df["strategy_return"] = df["return"] * df["position"]


# Time Invested

days_invested = (df["position"] == 1).sum()
valid_days = df["position"].count()
time_invested = days_invested / valid_days


# Trade Counts

position = df["position"].fillna(0).tolist()
trade_indicator = [0]

for i in range(1, len(position)):
    if position[i] != position[i-1]:
        trade_indicator.append(1)
    else:
        trade_indicator.append(0)

df["trade"] = trade_indicator
trade_count = df["trade"].sum()


# Transaction Costs

df["transaction_cost"] = df["trade"] * transaction_cost
df["net_strategy_return"] = df["strategy_return"] - df["transaction_cost"]


# Equity

performance_return = df["net_strategy_return"]

initial_equity = 1.0
df.loc[0, "equity"] = initial_equity

for i in range(1,len(df["equity"])):
    df.loc[i, "equity"] = df.loc[i-1, "equity"] * (1 + performance_return[i])

final_equity = float(df.loc[len(df)-1, "equity"])
total_return = final_equity/initial_equity - 1


# Volatility + Sharpe Ratio

risk_free_rate = 0

daily_volatility = performance_return.std()
mean_daily_return = performance_return.mean()
sharpe_daily = (mean_daily_return - risk_free_rate)/daily_volatility

annual_volatility = np.sqrt(252) * daily_volatility
sharpe_ratio = np.sqrt(252) * sharpe_daily


# Drawdown

df.loc[0, "running_peak"] = df.loc[0, "equity"]

for i in range(1, len(df["equity"])):
    df.loc[i, "running_peak"] = max(df.loc[i, "equity"], df.loc[i-1, "running_peak"])

df["drawdown"] = df["equity"] / df["running_peak"] - 1
max_drawdown = df["drawdown"].min()


# CAGR + Calmar Ratio

growth_multiple = final_equity / initial_equity
years = performance_return.count() / 252
cagr = growth_multiple ** (1 / years) - 1
calmar_ratio = cagr / abs(max_drawdown)


results = {
    "total_return": total_return,
    "cagr": cagr,
    "annual_volatility": annual_volatility,
    "annual_sharpe": sharpe_ratio,
    "max_drawdown": max_drawdown,
    "calmar": calmar_ratio,
    "time_invested": time_invested,
    "trades": trade_count
}

3.11 ms ± 168 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
backtest_df, results = backtest(price, signal, 0.001)

display(backtest_df)
display(pd.Series(results))

,return,signal,position,strategy_return,trade,transaction_cost,net_strategy_return,equity,running_peak,drawdown
0,NaN,0,NaN,NaN,0,0.000,NaN,1.000000,1.000000,0.000000
1,0.500000,1,0.0,0.000000,0,0.000,0.000000,1.000000,1.000000,0.000000
2,-0.200000,1,1.0,-0.200000,1,0.001,-0.201000,0.799000,1.000000,-0.201000
3,0.083333,0,1.0,0.083333,0,0.000,0.083333,0.865583,1.000000,-0.134417
4,0.538462,1,0.0,0.000000,1,0.001,-0.001000,0.864718,1.000000,-0.135282
5,0.050000,0,1.0,0.050000,1,0.001,0.049000,0.907089,1.000000,-0.092911
6,-0.142857,0,0.0,-0.000000,1,0.001,-0.001000,0.906182,1.000000,-0.093818
7,0.055556,1,0.0,0.000000,0,0.000,0.000000,0.906182,1.000000,-0.093818
8,0.315789,1,1.0,0.315789,1,0.001,0.314789,1.191438,1.191438,0.000000
9,-0.120000,1,1.0,-0.120000,0,0.000,-0.120000,1.048466,1.191438,-0.120000


total_return          0.048466
cagr                  2.762866
annual_volatility     2.255696
annual_sharpe         1.540739
max_drawdown         -0.201000
calmar               13.745602
time_invested         0.555556
trades                5.000000
dtype: float64

#### Create a little dataset

In [8]:
data = {
    "date": ["13/6", "14/6", "15/6", "16/6", "17/6", "18/6", "19/6", "20/6", "21/6", "22/6"],
    "price": [10, 15, 12, 13, 20, 21, 18, 19, 25, 22]
}

df = pd.DataFrame(data)

#### Calculate Returns

In [9]:
def calculate_return(price):

    returns = [None]

    for i in range(1,len(price)):
        daily_return = (price[i] - price[i-1])/price[i-1]
        returns.append(daily_return)

    return returns

In [10]:
df["return"] = calculate_return(df["price"])

#### Trading Signal & Strategy Return

In [11]:
df["signal"] = [0, 1, 1, 0, 1, 0, 0, 1, 1, 1]

df["position"] = df["signal"].shift(1)

df["strategy_return"] = df["return"] * df["position"]
df

,date,price,return,signal,position,strategy_return
0,13/6,10,NaN,0,NaN,NaN
1,14/6,15,0.500000,1,0.0,0.000000
2,15/6,12,-0.200000,1,1.0,-0.200000
3,16/6,13,0.083333,0,1.0,0.083333
4,17/6,20,0.538462,1,0.0,0.000000
5,18/6,21,0.050000,0,1.0,0.050000
6,19/6,18,-0.142857,0,0.0,-0.000000
7,20/6,19,0.055556,1,0.0,0.000000
8,21/6,25,0.315789,1,1.0,0.315789
9,22/6,22,-0.120000,1,1.0,-0.120000


#### Equity Curve

In [12]:
initial_equity = 1.0
df.loc[0, "equity"] = initial_equity

for i in range(1,len(df["equity"])):
    df.loc[i, "equity"] = df.loc[i-1, "equity"] * (1 + df.loc[i, "strategy_return"])

df

,date,price,return,signal,position,strategy_return,equity
0,13/6,10,NaN,0,NaN,NaN,1.000000
1,14/6,15,0.500000,1,0.0,0.000000,1.000000
2,15/6,12,-0.200000,1,1.0,-0.200000,0.800000
3,16/6,13,0.083333,0,1.0,0.083333,0.866667
4,17/6,20,0.538462,1,0.0,0.000000,0.866667
5,18/6,21,0.050000,0,1.0,0.050000,0.910000
6,19/6,18,-0.142857,0,0.0,-0.000000,0.910000
7,20/6,19,0.055556,1,0.0,0.000000,0.910000
8,21/6,25,0.315789,1,1.0,0.315789,1.197368
9,22/6,22,-0.120000,1,1.0,-0.120000,1.053684


In [13]:
final_equity = float(df.loc[len(df)-1, "equity"])

total_return = final_equity/initial_equity - 1

print("Total Return:", round(total_return, 4))

Total Return: 0.0537


#### Volatility & Sharpe Ratio

In [14]:
risk_free_rate = 0

daily_volatility = df["strategy_return"].std()
mean_daily_return = df["strategy_return"].mean()
sharpe_daily = (mean_daily_return - risk_free_rate)/daily_volatility


annual_volatility = np.sqrt(252) * daily_volatility
sharpe_ratio = np.sqrt(252) * sharpe_daily

print("Sharpe Ratio:", round(sharpe_ratio, 4))
print("Volatility:", round(annual_volatility, 4))

Sharpe Ratio: 1.6019
Volatility: 2.257


#### Drawdown

In [15]:
df.loc[0, "running_peak"] = df.loc[0, "equity"]

for i in range(1, len(df["equity"])):
    df.loc[i, "running_peak"] = max(df.loc[i, "equity"], df.loc[i-1, "running_peak"])

df["drawdown"] = df["equity"] / df["running_peak"] - 1
print(df)

max_drawdown = df["drawdown"].min()
print("Max Drawdown:", round(max_drawdown, 4))

   date  price    return  signal  position  strategy_return    equity  \
0  13/6     10       NaN       0       NaN              NaN  1.000000   
1  14/6     15  0.500000       1       0.0         0.000000  1.000000   
2  15/6     12 -0.200000       1       1.0        -0.200000  0.800000   
3  16/6     13  0.083333       0       1.0         0.083333  0.866667   
4  17/6     20  0.538462       1       0.0         0.000000  0.866667   
5  18/6     21  0.050000       0       1.0         0.050000  0.910000   
6  19/6     18 -0.142857       0       0.0        -0.000000  0.910000   
7  20/6     19  0.055556       1       0.0         0.000000  0.910000   
8  21/6     25  0.315789       1       1.0         0.315789  1.197368   
9  22/6     22 -0.120000       1       1.0        -0.120000  1.053684   

   running_peak  drawdown  
0      1.000000  0.000000  
1      1.000000  0.000000  
2      1.000000 -0.200000  
3      1.000000 -0.133333  
4      1.000000 -0.133333  
5      1.000000 -0.090000  


#### CAGR & Calmar Ratio

In [16]:
growth_multiple = final_equity / initial_equity

years = df["strategy_return"].count() / 252

cagr = growth_multiple ** (1 / years) - 1

calmar_ratio = cagr / abs(max_drawdown)

print("CAGR:", round(cagr, 4))
print("Calmar Ratio:", round(calmar_ratio, 4))

CAGR: 3.3241
Calmar Ratio: 16.6204


#### Time Invested, Trades Count, Transaction Costs

In [17]:
days_invested = (df["position"] == 1).sum()

valid_days = df["position"].count()

time_invested = days_invested / valid_days

print("Time Invested:", round(time_invested, 4))

Time Invested: 0.5556


In [18]:
position = df["position"].fillna(0).tolist()

trade_indicator = [0]

for i in range(1, len(position)):
    if position[i] != position[i-1]:
        trade_indicator.append(1)
    else:
        trade_indicator.append(0)

df["trade"] = trade_indicator

trade_count = df["trade"].sum()
print("Trade Count:", trade_count)

Trade Count: 5


In [19]:
transaction_cost = 0.001

df["transaction_cost"] = df["trade"] * transaction_cost

df["net_strategy_return"] = df["strategy_return"] - df["transaction_cost"]

#### Net Versions

In [20]:
performance_return = df["net_strategy_return"]

In [21]:
initial_equity = 1.0
df.loc[0, "equity"] = initial_equity

for i in range(1,len(df["equity"])):
    df.loc[i, "equity"] = df.loc[i-1, "equity"] * (1 + performance_return[i])

In [22]:
final_equity = float(df.loc[len(df)-1, "equity"])

total_return = final_equity/initial_equity - 1

print("Total Return:", round(total_return, 4))

Total Return: 0.0485


In [23]:
risk_free_rate = 0

daily_volatility = performance_return.std()
mean_daily_return = performance_return.mean()
sharpe_daily = (mean_daily_return - risk_free_rate)/daily_volatility


annual_volatility = np.sqrt(252) * daily_volatility
sharpe_ratio = np.sqrt(252) * sharpe_daily

print("Sharpe Ratio:", round(sharpe_ratio, 4))
print("Volatility:", round(annual_volatility, 4))

Sharpe Ratio: 1.5407
Volatility: 2.2557


In [24]:
df.loc[0, "running_peak"] = df.loc[0, "equity"]

for i in range(1, len(df["equity"])):
    df.loc[i, "running_peak"] = max(df.loc[i, "equity"], df.loc[i-1, "running_peak"])

df["drawdown"] = df["equity"] / df["running_peak"] - 1
print(df)

max_drawdown = df["drawdown"].min()
print("Max Drawdown:", round(max_drawdown, 4))

   date  price    return  signal  position  strategy_return    equity  \
0  13/6     10       NaN       0       NaN              NaN  1.000000   
1  14/6     15  0.500000       1       0.0         0.000000  1.000000   
2  15/6     12 -0.200000       1       1.0        -0.200000  0.799000   
3  16/6     13  0.083333       0       1.0         0.083333  0.865583   
4  17/6     20  0.538462       1       0.0         0.000000  0.864718   
5  18/6     21  0.050000       0       1.0         0.050000  0.907089   
6  19/6     18 -0.142857       0       0.0        -0.000000  0.906182   
7  20/6     19  0.055556       1       0.0         0.000000  0.906182   
8  21/6     25  0.315789       1       1.0         0.315789  1.191438   
9  22/6     22 -0.120000       1       1.0        -0.120000  1.048466   

   running_peak  drawdown  trade  transaction_cost  net_strategy_return  
0      1.000000  0.000000      0             0.000                  NaN  
1      1.000000  0.000000      0             0.0

In [25]:
growth_multiple = final_equity / initial_equity

years = performance_return.count() / 252

cagr = growth_multiple ** (1 / years) - 1

calmar_ratio = cagr / abs(max_drawdown)

print("CAGR:", round(cagr, 4))
print("Calmar Ratio:", round(calmar_ratio, 4))

CAGR: 2.7629
Calmar Ratio: 13.7456
